In [2]:
import pandas as pd
import numpy as np
import torch
import datasets
from datasets import Dataset, Audio
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate
from sklearn.model_selection import train_test_split
from dataclasses import dataclass
from typing import List, Dict

# Prevent HuggingFace fingerprint crash
datasets.config.IN_MEMORY_MAX_SIZE = 1e9
datasets.config.USE_MEMOIZATION = False


In [3]:
print(" CUDA available?", torch.cuda.is_available())
print(" Using device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU, sadly.")


 CUDA available? True
 Using device: NVIDIA GeForce RTX 4090 Laptop GPU


In [3]:
files = [
    ("data/spain_female_1.parquet", "spain"),
    ("data/spain_female_2.parquet", "spain"),
    ("data/spain_male_1.parquet", "spain"),
    ("data/spain_male_2.parquet", "spain"),
    ("data/mexico_female.parquet", "mexico"),
    ("data/mexico_male.parquet", "mexico"),
]

dfs = [pd.read_parquet(f).assign(label=label) for f, label in files]
df = pd.concat(dfs).reset_index(drop=True)

label2id = {"spain": 0, "mexico": 1}
id2label = {v: k for k, v in label2id.items()}
df["label"] = df["label"].map(label2id)

# Speaker-disjoint split
unique_speakers = df["speaker_id"].unique()
train_speakers, test_speakers = train_test_split(unique_speakers, test_size=0.2, random_state=42)
train_df = df[df["speaker_id"].isin(train_speakers)].reset_index(drop=True)
test_df = df[df["speaker_id"].isin(test_speakers)].reset_index(drop=True)


In [4]:
ds_train = Dataset.from_pandas(train_df, preserve_index=False).cast_column("audio", Audio(sampling_rate=16000))
ds_test = Dataset.from_pandas(test_df, preserve_index=False).cast_column("audio", Audio(sampling_rate=16000))


In [5]:
model_name = "facebook/wav2vec2-large-xlsr-53-spanish"
processor = Wav2Vec2Processor.from_pretrained(model_name)

def preprocess(batch):
    MAX_LENGTH = 160000
    audio = batch["audio"]["array"]
    if len(audio) > MAX_LENGTH:
        print(f"⚠️ Truncating long audio from {len(audio)} to {MAX_LENGTH} samples")
        audio = audio[:MAX_LENGTH]
    inputs = processor(audio, sampling_rate=16000)
    batch["input_values"] = inputs["input_values"][0]
    batch["label"] = int(batch["label"])
    return batch

ds_train = ds_train.map(preprocess)
ds_test = ds_test.map(preprocess)


C:\Users\swift\PycharmProjects\Spanish_Clean\.venv\Lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/597 [00:00<?, ? examples/s]

⚠️ Truncating long audio from 244800 to 160000 samples
⚠️ Truncating long audio from 224000 to 160000 samples
⚠️ Truncating long audio from 299200 to 160000 samples
⚠️ Truncating long audio from 258080 to 160000 samples
⚠️ Truncating long audio from 186720 to 160000 samples
⚠️ Truncating long audio from 211200 to 160000 samples
⚠️ Truncating long audio from 288000 to 160000 samples
⚠️ Truncating long audio from 304000 to 160000 samples
⚠️ Truncating long audio from 317600 to 160000 samples
⚠️ Truncating long audio from 194400 to 160000 samples
⚠️ Truncating long audio from 348800 to 160000 samples
⚠️ Truncating long audio from 273600 to 160000 samples
⚠️ Truncating long audio from 336000 to 160000 samples
⚠️ Truncating long audio from 326400 to 160000 samples
⚠️ Truncating long audio from 321600 to 160000 samples
⚠️ Truncating long audio from 224800 to 160000 samples
⚠️ Truncating long audio from 163200 to 160000 samples
⚠️ Truncating long audio from 224000 to 160000 samples
⚠️ Truncat

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

⚠️ Truncating long audio from 240000 to 160000 samples
⚠️ Truncating long audio from 169600 to 160000 samples
⚠️ Truncating long audio from 272000 to 160000 samples
⚠️ Truncating long audio from 174400 to 160000 samples
⚠️ Truncating long audio from 283200 to 160000 samples
⚠️ Truncating long audio from 297600 to 160000 samples
⚠️ Truncating long audio from 228800 to 160000 samples
⚠️ Truncating long audio from 323200 to 160000 samples
⚠️ Truncating long audio from 340800 to 160000 samples
⚠️ Truncating long audio from 390400 to 160000 samples
⚠️ Truncating long audio from 265600 to 160000 samples
⚠️ Truncating long audio from 208000 to 160000 samples
⚠️ Truncating long audio from 336000 to 160000 samples
⚠️ Truncating long audio from 278400 to 160000 samples
⚠️ Truncating long audio from 252800 to 160000 samples
⚠️ Truncating long audio from 276800 to 160000 samples
⚠️ Truncating long audio from 467200 to 160000 samples
⚠️ Truncating long audio from 316800 to 160000 samples
⚠️ Truncat

In [ ]:
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    label2id=label2id,
    id2label=id2label,
)
model.freeze_feature_encoder()

torch_device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(torch_device)
print(" Model is on:", next(model.parameters()).device)


In [7]:
from sklearn.metrics import classification_report, confusion_matrix

def compute_metrics(pred):
    preds = np.argmax(pred.predictions, axis=1)
    labels = pred.label_ids
    report = classification_report(labels, preds, target_names=["spain", "mexico"], output_dict=True)
    cm = confusion_matrix(labels, preds)

    per_class_acc = cm.diagonal() / cm.sum(axis=1)

    return {
        "accuracy": report["accuracy"],
        "precision_spain": report["spain"]["precision"],
        "recall_spain": report["spain"]["recall"],
        "f1_spain": report["spain"]["f1-score"],
        "support_spain": report["spain"]["support"],

        "precision_mexico": report["mexico"]["precision"],
        "recall_mexico": report["mexico"]["recall"],
        "f1_mexico": report["mexico"]["f1-score"],
        "support_mexico": report["mexico"]["support"],

        "f1_avg": report["weighted avg"]["f1-score"]
    }


In [8]:
@dataclass
class DataCollatorWithPadding:
    processor: Wav2Vec2Processor

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_values = [f["input_values"] for f in features]
        labels = torch.tensor([f["label"] for f in features])
        batch = self.processor.pad(
            {"input_values": input_values},
            padding=True,
            return_tensors="pt"
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorWithPadding(processor=processor)


In [9]:
training_args = TrainingArguments(
    output_dir="./wav2vec2-dialect",
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    num_train_epochs=10,
    fp16=False,
    dataloader_num_workers=0,
    report_to="none",
)



In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

trainer.train()


C:\Users\swift\PycharmProjects\Spanish_Clean\.venv\Lib\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision Spain,Recall Spain,F1 Spain,Support Spain,Precision Mexico,Recall Mexico,F1 Mexico,Support Mexico,F1 Avg
1,0.590700,0.357217,0.858639,0.766667,0.534884,0.630137,43.000000,0.875776,0.952703,0.912621,148.000000,0.849025
2,0.039700,0.483960,0.874346,0.720930,0.720930,0.720930,43.000000,0.918919,0.918919,0.918919,148.000000,0.874346
3,0.032500,0.994689,0.806283,0.542857,0.883721,0.672566,43.000000,0.958678,0.783784,0.862454,148.000000,0.819704
4,0.014200,0.889761,0.827225,0.578125,0.860465,0.691589,43.000000,0.952756,0.817568,0.880000,148.000000,0.837583
5,0.001700,0.499055,0.926702,0.853659,0.813953,0.833333,43.000000,0.946667,0.959459,0.953020,148.000000,0.926075
6,0.014600,0.682481,0.890052,0.892857,0.581395,0.704225,43.000000,0.889571,0.979730,0.932476,148.000000,0.881090
7,0.036600,1.057370,0.832461,0.587302,0.860465,0.698113,43.000000,0.953125,0.824324,0.884058,148.000000,0.842196
8,0.000300,0.701940,0.900524,0.900000,0.627907,0.739726,43.000000,0.900621,0.979730,0.938511,148.000000,0.893759
9,0.000400,0.571228,0.921466,0.868421,0.767442,0.814815,43.000000,0.934641,0.966216,0.950166,148.000000,0.919694
10,0.001000,0.567536,0.921466,0.868421,0.767442,0.814815,43.000000,0.934641,0.966216,0.950166,148.000000,0.919694


TrainOutput(global_step=2990, training_loss=0.07318211255165247, metrics={'train_runtime': 1334.6906, 'train_samples_per_second': 4.473, 'train_steps_per_second': 2.24, 'total_flos': 1.8075883473470016e+18, 'train_loss': 0.07318211255165247, 'epoch': 10.0})

In [11]:
results = trainer.evaluate()
print("Final evaluation:", results)


Final evaluation: {'eval_loss': 0.5675358176231384, 'eval_accuracy': 0.9214659685863874, 'eval_precision_spain': 0.868421052631579, 'eval_recall_spain': 0.7674418604651163, 'eval_f1_spain': 0.8148148148148148, 'eval_support_spain': 43.0, 'eval_precision_mexico': 0.934640522875817, 'eval_recall_mexico': 0.9662162162162162, 'eval_f1_mexico': 0.9501661129568106, 'eval_support_mexico': 148.0, 'eval_f1_avg': 0.9196943547363613, 'eval_runtime': 21.7106, 'eval_samples_per_second': 8.798, 'eval_steps_per_second': 4.422, 'epoch': 10.0}
